# Phase 1 — Probability Foundations

This notebook visualises the key distributions used in the budget model and
verifies the analytical moments derived in `notes/phase1-probability.md`.

## Distributions in the Budget Model

| Component | Variable | Distribution |
|-----------|----------|--------------|
| Salary | $S_i$ | LogNormal(9.2, 0.3²) |
| Overtime hours | $H_i$ | Poisson(5) |
| Incident count | $I$ | Poisson(3) |
| Incident cost | $C_j$ | LogNormal(10.5, 0.5²) |

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
rng = np.random.default_rng(42)

## 1. Salary Distribution — LogNormal(9.2, 0.3²)

Salaries are modelled as LogNormal because they are strictly positive and
right-skewed (a few high earners pull the mean above the median).

In [ ]:
# Parameters
mu_s, sigma_s = 9.2, 0.3

# Analytical moments
E_S = np.exp(mu_s + sigma_s**2 / 2)
Var_S = (np.exp(sigma_s**2) - 1) * np.exp(2 * mu_s + sigma_s**2)
SD_S = np.sqrt(Var_S)
median_S = np.exp(mu_s)

print(f"Analytical E[S]   = R$ {E_S:,.2f}")
print(f"Analytical Var[S] = {Var_S:,.2f}")
print(f"Analytical SD[S]  = R$ {SD_S:,.2f}")
print(f"Median[S]         = R$ {median_S:,.2f}")
print(f"Mean/Median ratio = {E_S / median_S:.4f} (>1 confirms right-skew)")

In [ ]:
# Sample and verify
n_samples = 100_000
salary_samples = rng.lognormal(mean=mu_s, sigma=sigma_s, size=n_samples)

print(f"\nEmpirical E[S]   = R$ {salary_samples.mean():,.2f}  (analytical: R$ {E_S:,.2f})")
print(f"Empirical SD[S]  = R$ {salary_samples.std():,.2f}  (analytical: R$ {SD_S:,.2f})")
print(f"Empirical Median = R$ {np.median(salary_samples):,.2f}  (analytical: R$ {median_S:,.2f})")

In [ ]:
# Plot: PDF + histogram
fig, ax = plt.subplots(figsize=(10, 5))

x = np.linspace(1_000, 30_000, 1000)
pdf_vals = stats.lognorm.pdf(x, s=sigma_s, scale=np.exp(mu_s))

ax.hist(salary_samples, bins=100, density=True, alpha=0.5, color="steelblue",
        label="Empirical (100K samples)", range=(1_000, 30_000))
ax.plot(x, pdf_vals, "darkblue", lw=2, label="Analytical PDF")
ax.axvline(E_S, color="red", ls="--", lw=1.5, label=f"E[S] = R$ {E_S:,.0f}")
ax.axvline(median_S, color="orange", ls="--", lw=1.5,
           label=f"Median = R$ {median_S:,.0f}")

ax.set_xlabel("Monthly Salary (R$)")
ax.set_ylabel("Density")
ax.set_title("Salary Distribution — LogNormal(9.2, 0.09)")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/dist_salary_lognormal.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Overtime Hours — Poisson(5)

Overtime hours per employee per month follow a Poisson distribution.
The Poisson models count data where events occur independently at a constant rate.

In [ ]:
# Parameters
lambda_h = 5

# Analytical: E[H] = Var[H] = lambda
print(f"Analytical E[H]   = {lambda_h}")
print(f"Analytical Var[H] = {lambda_h}")

# Sample and verify
overtime_samples = rng.poisson(lam=lambda_h, size=n_samples)
print(f"\nEmpirical E[H]   = {overtime_samples.mean():.4f}")
print(f"Empirical Var[H] = {overtime_samples.var():.4f}")

In [ ]:
# Plot: PMF + histogram
fig, ax = plt.subplots(figsize=(10, 5))

k_vals = np.arange(0, 20)
pmf_vals = stats.poisson.pmf(k_vals, mu=lambda_h)

ax.bar(k_vals - 0.2, pmf_vals, width=0.4, color="steelblue", alpha=0.7,
       label="Analytical PMF")

# Empirical histogram
counts = np.bincount(overtime_samples, minlength=20)[:20]
empirical_pmf = counts / n_samples
ax.bar(k_vals + 0.2, empirical_pmf, width=0.4, color="coral", alpha=0.7,
       label="Empirical (100K samples)")

ax.axvline(lambda_h, color="red", ls="--", lw=1.5,
           label=f"E[H] = Var[H] = {lambda_h}")

ax.set_xlabel("Overtime Hours per Month")
ax.set_ylabel("Probability")
ax.set_title("Overtime Distribution — Poisson(5)")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/dist_overtime_poisson.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Incident Count — Poisson(3)

Severe incidents per year follow Poisson(3). These are rare, disruptive events
that cause additional cost beyond regular operations.

In [ ]:
lambda_I = 3

print(f"Analytical E[I]   = {lambda_I}")
print(f"Analytical Var[I] = {lambda_I}")

incident_count_samples = rng.poisson(lam=lambda_I, size=n_samples)
print(f"\nEmpirical E[I]   = {incident_count_samples.mean():.4f}")
print(f"Empirical Var[I] = {incident_count_samples.var():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

k_vals = np.arange(0, 15)
pmf_vals = stats.poisson.pmf(k_vals, mu=lambda_I)

ax.bar(k_vals - 0.2, pmf_vals, width=0.4, color="steelblue", alpha=0.7,
       label="Analytical PMF")

counts = np.bincount(incident_count_samples, minlength=15)[:15]
empirical_pmf = counts / n_samples
ax.bar(k_vals + 0.2, empirical_pmf, width=0.4, color="coral", alpha=0.7,
       label="Empirical (100K samples)")

ax.axvline(lambda_I, color="red", ls="--", lw=1.5,
           label=f"E[I] = Var[I] = {lambda_I}")

ax.set_xlabel("Number of Severe Incidents per Year")
ax.set_ylabel("Probability")
ax.set_title("Incident Count — Poisson(3)")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/dist_incidents_poisson.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Incident Cost — LogNormal(10.5, 0.5²)

Each severe incident has a cost drawn from LogNormal(10.5, 0.25). The high
variance reflects that incidents range from minor disruptions to major outages.

In [ ]:
mu_I, sigma_I = 10.5, 0.5

E_C = np.exp(mu_I + sigma_I**2 / 2)
Var_C = (np.exp(sigma_I**2) - 1) * np.exp(2 * mu_I + sigma_I**2)
SD_C = np.sqrt(Var_C)
median_C = np.exp(mu_I)

print(f"Analytical E[C]   = R$ {E_C:,.2f}")
print(f"Analytical SD[C]  = R$ {SD_C:,.2f}")
print(f"Median[C]         = R$ {median_C:,.2f}")

cost_samples = rng.lognormal(mean=mu_I, sigma=sigma_I, size=n_samples)
print(f"\nEmpirical E[C]   = R$ {cost_samples.mean():,.2f}")
print(f"Empirical SD[C]  = R$ {cost_samples.std():,.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.linspace(1_000, 200_000, 1000)
pdf_vals = stats.lognorm.pdf(x, s=sigma_I, scale=np.exp(mu_I))

ax.hist(cost_samples, bins=150, density=True, alpha=0.5, color="steelblue",
        label="Empirical (100K samples)", range=(1_000, 200_000))
ax.plot(x, pdf_vals, "darkblue", lw=2, label="Analytical PDF")
ax.axvline(E_C, color="red", ls="--", lw=1.5, label=f"E[C] = R$ {E_C:,.0f}")
ax.axvline(median_C, color="orange", ls="--", lw=1.5,
           label=f"Median = R$ {median_C:,.0f}")

ax.set_xlabel("Incident Cost (R$)")
ax.set_ylabel("Density")
ax.set_title("Incident Cost Distribution — LogNormal(10.5, 0.25)")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/dist_incident_cost_lognormal.png", dpi=150,
            bbox_inches="tight")
plt.show()

## 5. Verification of Budget Model Analytical Moments

We compute the analytical $E[X_{\text{total}}]$ and $\text{Var}(X_{\text{total}})$
and verify them against a large sample simulation.

In [ ]:
# === Model parameters ===
n_employees = 50
mu_s, sigma_s = 9.2, 0.3
beta = 1.80
lambda_h = 5
r_ot = 80
lambda_I = 3
mu_I, sigma_I = 10.5, 0.5

# === Analytical moments ===
# Term 1: Salaries
E_S = np.exp(mu_s + sigma_s**2 / 2)
Var_S = (np.exp(sigma_s**2) - 1) * np.exp(2 * mu_s + sigma_s**2)

E_T1 = n_employees * beta * 12 * E_S
Var_T1 = n_employees * (beta * 12)**2 * Var_S

# Term 2: Overtime
E_T2 = n_employees * r_ot * 12 * lambda_h
Var_T2 = n_employees * (r_ot * 12)**2 * lambda_h  # Var[H] = lambda_h

# Term 3: Incidents (compound Poisson)
E_C = np.exp(mu_I + sigma_I**2 / 2)
E_C2 = np.exp(2 * mu_I + 2 * sigma_I**2)

E_T3 = lambda_I * E_C
Var_T3 = lambda_I * E_C2  # Var(compound Poisson) = lambda * E[C^2]

# Total
E_total = E_T1 + E_T2 + E_T3
Var_total = Var_T1 + Var_T2 + Var_T3
SD_total = np.sqrt(Var_total)

print("=== Analytical Moments ===")
print(f"E[Term 1]  = R$ {E_T1:>15,.2f}  ({E_T1/E_total*100:.1f}%)")
print(f"E[Term 2]  = R$ {E_T2:>15,.2f}  ({E_T2/E_total*100:.1f}%)")
print(f"E[Term 3]  = R$ {E_T3:>15,.2f}  ({E_T3/E_total*100:.1f}%)")
print(f"E[X_total] = R$ {E_total:>15,.2f}")
print(f"SD[X_total]= R$ {SD_total:>15,.2f}")
print()
print(f"Var[Term 1] = {Var_T1:.4e}  ({Var_T1/Var_total*100:.1f}%)")
print(f"Var[Term 2] = {Var_T2:.4e}  ({Var_T2/Var_total*100:.1f}%)")
print(f"Var[Term 3] = {Var_T3:.4e}  ({Var_T3/Var_total*100:.1f}%)")

In [ ]:
def simulate_one_year(rng: np.random.Generator) -> float:
    """Simulate one year of the budget model and return total cost."""
    # Term 1: salaries
    salaries = rng.lognormal(mean=mu_s, sigma=sigma_s, size=n_employees)
    term1 = np.sum(salaries) * beta * 12

    # Term 2: overtime (sum over employees and months)
    overtime_hours = rng.poisson(lam=lambda_h, size=(n_employees, 12))
    term2 = np.sum(overtime_hours) * r_ot

    # Term 3: incidents
    n_incidents = rng.poisson(lam=lambda_I)
    if n_incidents > 0:
        incident_costs = rng.lognormal(mean=mu_I, sigma=sigma_I,
                                       size=n_incidents)
        term3 = np.sum(incident_costs)
    else:
        term3 = 0.0

    return term1 + term2 + term3

In [ ]:
# Run simulation
n_sims = 50_000
sim_rng = np.random.default_rng(42)
results = np.array([simulate_one_year(sim_rng) for _ in range(n_sims)])

print("=== Empirical vs Analytical ===")
print(f"Empirical  E[X] = R$ {results.mean():>15,.2f}")
print(f"Analytical E[X] = R$ {E_total:>15,.2f}")
print(f"Difference      =     {abs(results.mean() - E_total)/E_total*100:.3f}%")
print()
print(f"Empirical  SD[X] = R$ {results.std():>14,.2f}")
print(f"Analytical SD[X] = R$ {SD_total:>14,.2f}")

In [ ]:
# Distribution overview plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(results / 1e6, bins=80, density=True, alpha=0.6,
             color="steelblue", edgecolor="white")
axes[0].axvline(E_total / 1e6, color="red", ls="--", lw=2,
               label=f"Analytical E[X] = R$ {E_total/1e6:.2f}M")
axes[0].axvline(results.mean() / 1e6, color="darkgreen", ls=":", lw=2,
               label=f"Empirical mean = R$ {results.mean()/1e6:.2f}M")
axes[0].set_xlabel("Total Annual Cost (R$ millions)")
axes[0].set_ylabel("Density")
axes[0].set_title("Budget Cost Distribution (50K simulations)")
axes[0].legend(fontsize=9)

# Variance decomposition
labels = ["Salaries\n(Term 1)", "Overtime\n(Term 2)", "Incidents\n(Term 3)"]
var_shares = [Var_T1 / Var_total * 100, Var_T2 / Var_total * 100,
              Var_T3 / Var_total * 100]
colors = ["steelblue", "coral", "gold"]
axes[1].bar(labels, var_shares, color=colors, edgecolor="white")
axes[1].set_ylabel("% of Total Variance")
axes[1].set_title("Variance Decomposition by Component")
for i, v in enumerate(var_shares):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("../figures/budget_overview_phase1.png", dpi=150,
            bbox_inches="tight")
plt.show()

## Key Takeaways

1. **Empirical moments match analytical moments** — confirming the derivations
   in `notes/phase1-probability.md`.

2. **Salaries dominate** both expected value (~97%) and variance (~97%).
   Overtime and incidents are small relative contributors.

3. **The distribution is not symmetric** — the right-skew from LogNormal
   salaries means the mean is above the median. Point estimates using the
   mean alone miss this asymmetry.

4. **What we still cannot do with analytics alone:**
   - Compute exact $P(X > B)$ for a budget ceiling $B$
   - Build confidence intervals (requires CLT, Phase 3)
   - Assess sensitivity to distributional assumptions

   Monte Carlo simulation (Phase 4) will address all of these.